# Linking Health Datasets: Wisconsin Breast Cancer + SEER Registry

## Student Exercise Notebook

**Course:** I 320D - Data Science for Biomedical Informatics
**Instructor:** Ammar Darkazanli
**Semester:** Spring 2026

---

### 🎯 This Week's Mantra

# "One Patient, Many Datasets"

*Translation: No single dataset tells the whole story. Learning to combine data sources is a core real-world skill.*

---

## Instructions

1. Complete each cell marked with `# TODO:`
2. Run your code to verify it works
3. Use the hints provided if you get stuck
4. Ask questions during class if needed!

---

### Learning Objectives

By the end of this exercise, you will be able to:

1. Load and compare image-derived vs. registry-based cancer datasets
2. Harmonize features across datasets (tumor size units, target encoding, grade systems)
3. Combine datasets vertically with `pd.concat` — navigating incompatible feature spaces
4. Build lookup tables with `groupby` and enrich FNA records with population-level survival data
5. Use a trained model to generate survival-risk scores across datasets

---

### Notebook Sections
| Part | Topic | Strategy |
|------|-------|----------|
| 1 | Setup and Load Data | — |
| 2 | Schema Harmonization | — |
| 3 | Vertical Stack | `pd.concat` |
| 4 | Aggregate Enrichment | `groupby` + `merge` |
| 5 | Train-Transfer | Model as Bridge |
| 6 | Challenge Exercises | All strategies |

---

### 📋 Dataset Overview

| | Wisconsin BC (Diagnostic) | SEER Breast Cancer Registry |
|---|---|---|
| **Source** | UCI / sklearn | NCI SEER Program |
| **Rows** | 569 | ~4,024 |
| **Features** | 30 numeric (FNA image measurements) | 15 mixed (clinical staging, demographics, survival) |
| **Target** | `diagnosis` (M/B) — malignancy | `Status` (Alive/Dead) — survival |
| **Modality** | Fine Needle Aspirate imaging | Population cancer registry |
| **Key bridge** | `radius_mean` (image-derived) | `Tumor Size` (pathology, mm) |

---
# PART 1: Setup and Load Data
---

### 1.1 Import Libraries

In [ ]:
# Importing the required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Importing Wisconsin Breast Cancer
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()

# Display settings (run this after importing)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-whitegrid')

import warnings
warnings.filterwarnings('ignore')

### 1.2 Load the Datasets

> **Wisconsin BC:** Built into sklearn — no download needed!
> **SEER:** Download from kaggle.com/datasets/reihanenamdari/breast-cancer
> Use the file: `Breast_Cancer.csv`

In [ ]:
# Loading both datasets
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
wbc = pd.DataFrame(data.data, columns=data.feature_names)
wbc['diagnosis'] = data.target  # 0=malignant, 1=benign

seer = pd.read_csv('Breast_Cancer.csv')

print(f"Wisconsin BC: {wbc.shape[0]:,} rows x {wbc.shape[1]} columns")
print(f"SEER:         {seer.shape[0]:,} rows x {seer.shape[1]} columns")

Wisconsin BC: 569 rows x 31 columns
SEER:         4,024 rows x 16 columns


### 1.3 Explore Both Datasets

In [ ]:
# Display the first 3 rows of the Wisconsin BC dataset
wbc.head(3)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis
0,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0


In [ ]:
# Display the first 3 rows of the SEER dataset
seer.head(3)

,Age,Race,Marital Status,T Stage,N Stage,6th Stage,differentiate,Grade,A Stage,Tumor Size,Estrogen Status,Progesterone Status,Regional Node Examined,Reginol Node Positive,Survival Months,Status
0,68,White,Married,T1,N1,IIA,Poorly differentiated,3,Regional,4,Positive,Positive,24,1,60,Alive
1,50,White,Married,T2,N2,IIIA,Moderately differentiated,2,Regional,35,Positive,Positive,14,5,62,Alive
2,58,White,Divorced,T3,N3,IIIC,Moderately differentiated,2,Regional,63,Positive,Positive,14,7,75,Alive


In [ ]:
#Displaying the columns and their information
print("WISCONSIN BC COLUMNS:")
wbc.info()

print("\nSEER COLUMNS:")
seer.info()

WISCONSIN BC COLUMNS:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoo

### 📋 Feature Alignment Map

Before harmonizing, identify which columns correspond between datasets.

| Feature | Wisconsin Column | Wisconsin Format | SEER Column | SEER Format | Harmonization Needed |
|---------|-----------------|-----------------|-------------|-------------|---------------------|
| Tumor Size | `radius_mean` | Image-derived float (unitless) | `Tumor Size` | Pathology (mm, int) | **Cannot convert directly** — different modalities |
| Diagnosis | `diagnosis` | 0=Malignant, 1=Benign | `Status` | "Alive"/"Dead" strings | Different outcomes entirely |
| — | — | — | `T Stage` | "T1"/"T2"/"T3"/"T4" text | Encode to ordinal |
| — | — | — | `N Stage` | "N1"/"N2"/"N3" text | Encode to ordinal |
| — | — | — | `Grade` | Text labels | Encode to ordinal |
| — | — | — | `Estrogen Status` | "Positive"/"Negative" | Encode to binary |
| — | — | — | `Progesterone Status` | "Positive"/"Negative" | Encode to binary |
| — | — | — | `Age` | Integer years | Create bins |
| Texture | `texture_mean` | Float | — | Not available | — |
| Compactness | `compactness_mean` | Float | — | Not available | — |

> ⚠️ **Key Insight:** These datasets measure **fundamentally different things** about breast cancer.
> Wisconsin captures **cell morphology** from microscope images (shape, size, texture of nuclei).
> SEER captures **clinical staging and outcomes** from cancer registries (TNM stage, receptor status, survival).
> The only physical bridge is tumor size — but measured in incompatible ways.

### SEER Grade Reference

| Grade | Description |
|-------|-------------|
| Well differentiated | Grade I — cells look nearly normal |
| Moderately differentiated | Grade II — cells look somewhat abnormal |
| Poorly differentiated | Grade III — cells look very abnormal |
| Undifferentiated | Grade IV — cells barely resemble normal tissue |

### Clinical Context: TNM Staging

| Stage | Meaning |
|-------|---------|
| T1–T4 | **Tumor** size/extent (T1 = small, T4 = large/invasive) |
| N1–N3 | **Node** involvement (N1 = few nodes, N3 = many nodes) |
| Regional/Distant | **Metastasis** extent |

### 1.4 Data Validation

In [ ]:
# TODO: Validate the Wisconsin BC dataset
# Check: missing values, diagnosis distribution (0=malignant, 1=benign),
#        radius_mean range, number of features
# Hint: wbc['diagnosis'].value_counts()

print("="*60)
print("WISCONSIN BC VALIDATION")
print("="*60)

# YOUR CODE HERE

In [ ]:
# TODO: Validate the SEER dataset
# Check: missing values, Status distribution, Age range,
#        T Stage categories, Grade categories, Tumor Size range
# Hint: seer['T Stage '].value_counts()
# Note: Some SEER column names may have trailing spaces!

print("="*60)
print("SEER DATASET VALIDATION")
print("="*60)

# YOUR CODE HERE

In [ ]:
# TODO: Clean SEER column names — strip whitespace
# Hint: seer.columns = seer.columns.str.strip()
# Then verify with seer.columns.tolist()

# YOUR CODE HERE
seer.columns = seer.columns.str.strip()

print("Cleaned SEER columns:")
print(seer.columns.tolist())

Cleaned SEER columns:
['Age', 'Race', 'Marital Status', 'T Stage', 'N Stage', '6th Stage', 'differentiate', 'Grade', 'A Stage', 'Tumor Size', 'Estrogen Status', 'Progesterone Status', 'Regional Node Examined', 'Reginol Node Positive', 'Survival Months', 'Status']


In [ ]:
print("="*60)
print("SEER DATASET VALIDATION")
print("="*60)

# YOUR CODE HERE
print("Missing values:")
print(seer.isnull().sum())

print("\nStatus distribution:")
print(seer['Status'].value_counts())

print("\nAge range:")
print(seer['Age'].describe())

print("\nT Stage categories:")
print(seer['T Stage'].value_counts())

print("\nGrade categories:")
print(seer['Grade'].value_counts())

print("\nTumor Size range:")
print(seer['Tumor Size'].describe())

SEER DATASET VALIDATION
Missing values:
Age                       0
Race                      0
Marital Status            0
T Stage                   0
N Stage                   0
6th Stage                 0
differentiate             0
Grade                     0
A Stage                   0
Tumor Size                0
Estrogen Status           0
Progesterone Status       0
Regional Node Examined    0
Reginol Node Positive     0
Survival Months           0
Status                    0
dtype: int64

Status distribution:
Status
Alive    3408
Dead      616
Name: count, dtype: int64

Age range:
count    4024.000000
mean       53.972167
std         8.963134
min        30.000000
25%        47.000000
50%        54.000000
75%        61.000000
max        69.000000
Name: Age, dtype: float64

T Stage categories:
T Stage
T2    1786
T1    1603
T3     533
T4     102
Name: count, dtype: int64

Grade categories:
Grade
2                        2351
3                        1111
1                         

---
# PART 2: Schema Harmonization
---

Before linking, we need to bring both datasets into a **common vocabulary**.

### 2.1 Recode Wisconsin Diagnosis to Strings

sklearn encodes diagnosis as 0=malignant, 1=benign. Create a string version for clarity.

In [ ]:
# Create 'diagnosis_str' column in wbc & Map: 0 -> 'Malignant', 1 -> 'Benign'
wbc['diagnosis_str'] = wbc['diagnosis'].map({0: 'Malignant', 1: 'Benign'})

print("Wisconsin Diagnosis:")
print(wbc['diagnosis_str'].value_counts())
print(f"Malignancy rate: {(wbc['diagnosis'] == 0).mean()*100:.1f}%")

Wisconsin Diagnosis:
diagnosis_str
Benign       357
Malignant    212
Name: count, dtype: int64
Malignancy rate: 37.3%


### 2.2 Encode SEER Survival Status to Binary

SEER uses "Alive"/"Dead" strings. Convert to numeric.

In [ ]:
# TODO: Create 'status_binary' in SEER: 'Dead' -> 1, 'Alive' -> 0

# YOUR CODE HERE
seer['status_binary'] = seer['Status'].map({'Dead': 1, 'Alive': 0})

print("SEER Status:")
print(seer['Status'].value_counts())
print(f"Mortality rate: {seer['status_binary'].mean()*100:.1f}%")

SEER Status:
Status
Alive    3408
Dead      616
Name: count, dtype: int64
Mortality rate: 15.3%


### 2.3 Encode SEER T Stage to Ordinal

Convert text T Stage labels to numeric ordinal values.

In [ ]:
# TODO: Create 't_stage_num' in SEER by mapping T Stage to numeric
# T1 -> 1, T2 -> 2, T3 -> 3, T4 -> 4
# Hint: seer['t_stage_num'] = seer['T Stage'].map({'T1': 1, 'T2': 2, 'T3': 3, 'T4': 4})

# YOUR CODE HERE

print("SEER T Stage (numeric):")
print(seer['t_stage_num'].value_counts().sort_index())

### 2.4 Encode SEER Grade to Ordinal

Convert text grade labels to numeric values for modeling.

In [ ]:
# TODO: Create 'grade_num' in SEER by mapping Grade to numeric
# Map text labels to 1, 2, 3, 4 (or use the first word)
# Hint: Check seer['Grade'].unique() first to see exact labels
# Then map: grade containing 'Well' -> 1, 'Moderately' -> 2,
#           'Poorly' -> 3, 'Undifferentiated' -> 4
# One approach: create a dictionary and use .map()

grade_map = {
    # YOUR CODE HERE -- fill in the mapping
    # e.g., 'Well differentiated; Grade I': 1
}

# If you're not sure of exact labels, try this first:
print("Unique grades:", seer['Grade'].unique())

# Then apply the mapping:
# seer['grade_num'] = seer['Grade'].map(grade_map)

### 2.5 Encode Receptor Status to Binary

Convert Estrogen and Progesterone status from strings to numeric.

In [ ]:
# TODO: Create binary columns for receptor status
# 'er_positive': 1 if Estrogen Status == 'Positive', else 0
# 'pr_positive': 1 if Progesterone Status == 'Positive', else 0

# YOUR CODE HERE

print("Estrogen Receptor Positive rate:", f"{seer['er_positive'].mean()*100:.1f}%")
print("Progesterone Receptor Positive rate:", f"{seer['pr_positive'].mean()*100:.1f}%")

### 2.6 Create Age Bins in SEER

SEER has patient age; Wisconsin does not. Create age bins for groupby operations.

In [ ]:
# TODO: Create 'age_bin' in SEER using pd.cut
# Bins: [0, 39, 49, 59, 69, 100]
# Labels: ['Under 40', '40-49', '50-59', '60-69', '70+']

bins = [0, 39, 49, 59, 69, 100]
labels = ['Under 40', '40-49', '50-59', '60-69', '70+']

seer['age_bin'] = # YOUR CODE HERE

print("SEER Age Bins:")
print(seer['age_bin'].value_counts().sort_index())

### 2.7 Create Tumor Size Bins

Since Wisconsin's `radius_mean` and SEER's `Tumor Size` use incompatible units, we can still create **ordinal size categories** to enable approximate linking.

> 💡 **Clinical context:** Tumor staging (T1–T4) is based on tumor diameter:
> - T1: ≤ 20 mm
> - T2: 21–50 mm
> - T3: > 50 mm
> - T4: Any size with chest wall/skin invasion
>
> We'll bin SEER's Tumor Size (mm) into T-stage-like categories.

In [ ]:
# TODO: Create 'tumor_size_cat' in SEER using pd.cut
# Bins: [0, 20, 50, 200]
# Labels: ['Small (≤20mm)', 'Medium (21-50mm)', 'Large (>50mm)']

seer['tumor_size_cat'] = # YOUR CODE HERE

print("SEER Tumor Size Categories:")
print(seer['tumor_size_cat'].value_counts().sort_index())

In [ ]:
# TODO: Create 'tumor_size_cat' in Wisconsin using radius_mean
# We can't convert directly, but we can create relative categories.
# Use the 33rd and 66th percentiles to split into 3 groups.
# Hint: Use pd.qcut(wbc['radius_mean'], q=3, labels=['Small', 'Medium', 'Large'])

wbc['tumor_size_cat'] = # YOUR CODE HERE

print("Wisconsin Tumor Size Categories (by percentile):")
print(wbc['tumor_size_cat'].value_counts().sort_index())

---
# PART 3: Strategy 1 — Vertical Stack (`pd.concat`)
---

**Goal:** Align shared columns, stack rows from both datasets into one DataFrame.

> ⚠️ **Important Caveat:** These datasets have very different feature spaces. The only truly shared feature is tumor size category (and even that uses different measurement scales). We stack them to practice the mechanics and to **discuss the limitations**.

### 📋 Shared columns to align:
- `tumor_size_cat` — both datasets (harmonized)
- `target` — Wisconsin: `diagnosis` (0=malignant, 1=benign), SEER: `status_binary` (0=alive, 1=dead)

> ⚠️ The targets measure **completely different outcomes**! Malignancy vs. survival. This stack is for **structural practice**.

In [ ]:
# TODO: Create wbc_aligned DataFrame with columns:
# tumor_size_cat, target (from diagnosis), dataset_source='wisconsin'

wbc_aligned = # YOUR CODE HERE

print("Wisconsin aligned:", wbc_aligned.shape)
wbc_aligned.head(3)

In [ ]:
# TODO: Create seer_aligned DataFrame with matching columns:
# tumor_size_cat, target (from status_binary), dataset_source='seer'

seer_aligned = # YOUR CODE HERE

print("SEER aligned:", seer_aligned.shape)
seer_aligned.head(3)

In [ ]:
# TODO: Stack both DataFrames using pd.concat
# Use ignore_index=True, then drop NaN rows with .dropna()

combined = # YOUR CODE HERE

print(f"Combined shape: {combined.shape[0]:,} rows")
print(f"\nBy dataset_source:")
print(combined['dataset_source'].value_counts())

In [ ]:
# TODO: Create a grouped bar chart showing target rate by tumor_size_cat and dataset_source
# Group by ['tumor_size_cat', 'dataset_source'], compute mean of 'target'
# Hint: Use .unstack().plot(kind='bar')

fig, ax = plt.subplots(figsize=(10, 5))

# YOUR CODE HERE


ax.set_title('Target Rate by Tumor Size Category and Source')
ax.set_xlabel('Tumor Size Category')
ax.set_ylabel('Target Rate')
ax.legend(title='Source')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Question:** The "target" means something completely different in each dataset (malignancy vs. death). Why does combining them into one bar chart create a misleading comparison? When might stacking datasets with different targets still be useful?

*Your answer:*

---
# PART 4: Strategy 2 — Aggregate Enrichment (`groupby` + `merge`)
---

**Goal:** Build a clinical-outcome lookup table from SEER data grouped by tumor size category, then merge it into Wisconsin BC to enrich each FNA sample with population-level survival statistics.

> 💡 **Why this works:** Even though the datasets measure different things, a Wisconsin patient with a "Large" tumor can be enriched with the population-level mortality rate, average node involvement, and receptor positivity rate for large-tumor SEER patients. This adds **clinical context** to morphological measurements.

### 4.1 Build the Clinical Lookup Table

In [ ]:
# TODO: Group the SEER dataset by 'tumor_size_cat' and compute:
# - mortality_rate: mean of status_binary
# - avg_tumor_size_mm: mean of Tumor Size (the actual column name in SEER)
# - avg_nodes_positive: mean of Reginol Node Positive (check exact column name!)
# - er_positive_rate: mean of er_positive
# - pr_positive_rate: mean of pr_positive
# - avg_survival_months: mean of Survival Months
# - n_patients: count of status_binary
# Then .round(3).reset_index()
#
# Hint: Check seer.columns to find exact column names for nodes and survival

lookup = # YOUR CODE HERE

print(f"Lookup table: {len(lookup)} tumor size groups")
display(lookup)

### 4.2 Merge Clinical Context into Wisconsin BC

In [ ]:
# TODO: Left merge the lookup table into wbc on 'tumor_size_cat'

wbc_enriched = # YOUR CODE HERE

print(f"Enriched shape: {wbc_enriched.shape}")
print(f"New columns: {[c for c in wbc_enriched.columns if c in lookup.columns and c != 'tumor_size_cat']}")
print(f"NaN in mortality_rate: {wbc_enriched['mortality_rate'].isnull().sum()}")

In [ ]:
# TODO: Compare enriched features between Malignant (0) and Benign (1)
# Group wbc_enriched by 'diagnosis' and show the mean of:
# mortality_rate, avg_nodes_positive, er_positive_rate, avg_survival_months

enriched_cols = ['mortality_rate', 'avg_nodes_positive', 'er_positive_rate', 'avg_survival_months']

# YOUR CODE HERE

In [ ]:
# TODO: Create a visualization comparing enriched clinical features by diagnosis
# Use a bar chart or heatmap showing the mean of enriched_cols for Malignant vs Benign

fig, ax = plt.subplots(figsize=(10, 5))

# YOUR CODE HERE


ax.set_title('Population-Level Clinical Profiles by Wisconsin Diagnosis')
plt.tight_layout()
plt.show()

**Question:** Do malignant Wisconsin tumors tend to fall in tumor size categories with higher mortality rates in the SEER registry? What does this tell us about the relationship between cell morphology and clinical outcomes? What are the limitations?

*Your answer:*

---
# PART 5: Strategy 3 — Train-Transfer (Model as Bridge)
---

**Goal:** Train a LogisticRegression on SEER data to predict mortality. Then use this model to generate a "mortality risk score" for each Wisconsin BC patient using their enriched features from Strategy 2.

> 💡 **The idea:** SEER teaches us which clinical profiles are associated with death. By enriching Wisconsin patients with population-level clinical data (Strategy 2) and then scoring them through a SEER-trained model (Strategy 3), we create a **cross-dataset risk estimate** that bridges imaging and registry data.

**Approach:**
1. Train the model on SEER: **tumor size + T stage + grade + receptor status → mortality**
2. Map Wisconsin patients to SEER-compatible features using enrichment + tumor size category
3. Generate mortality risk scores for Wisconsin patients

### 5.1 Prepare Training Data

In [ ]:
# TODO: Create X_seer with numeric columns from SEER:
# ['Tumor Size', 't_stage_num', 'grade_num', 'er_positive', 'pr_positive']
# Create y_seer from seer['status_binary']
# Drop any rows with NaN

train_features = ['Tumor Size', 't_stage_num', 'grade_num', 'er_positive', 'pr_positive']

X_seer = # YOUR CODE HERE
y_seer = # YOUR CODE HERE

# Drop NaN
mask = X_seer.notna().all(axis=1) & y_seer.notna()
X_seer = X_seer[mask]
y_seer = y_seer[mask]

print(f"Training data: {X_seer.shape[0]:,} rows, {X_seer.shape[1]} features")
print(f"Mortality rate: {y_seer.mean()*100:.1f}%")

### 5.2 Train the Model

In [ ]:
# TODO: Scale features with StandardScaler, then train LogisticRegression
# 1. Create scaler, fit_transform X_seer
# 2. Create model with random_state=42, max_iter=1000
# 3. Fit model on scaled data

scaler = # YOUR CODE HERE
X_seer_scaled = # YOUR CODE HERE

model = # YOUR CODE HERE
# YOUR CODE HERE -- fit the model

print(f"Training accuracy: {model.score(X_seer_scaled, y_seer)*100:.1f}%")

### 5.3 Generate Mortality Risk Scores for Wisconsin Patients

Since Wisconsin patients don't have staging, grade, or receptor status directly, we'll use their **enriched data** from Strategy 2 (population-level averages based on tumor size category).

In [ ]:
# TODO: Prepare Wisconsin data for prediction using enriched features
# Create X_wbc DataFrame with columns matching train_features:
#   'Tumor Size' from wbc_enriched['avg_tumor_size_mm']
#   't_stage_num' — map tumor_size_cat to approximate T stage:
#       'Small (≤20mm)' or 'Small' -> 1, 'Medium (21-50mm)' or 'Medium' -> 2,
#       'Large (>50mm)' or 'Large' -> 3
#   'grade_num' — we don't have grade, so use a neutral value (2.0) for all
#   'er_positive' from wbc_enriched['er_positive_rate']
#   'pr_positive' from wbc_enriched['pr_positive_rate']
#
# Drop NaN, scale using SAME scaler (transform, NOT fit_transform!), predict_proba

# Map tumor size to approximate T stage
size_to_tstage = {
    'Small (≤20mm)': 1, 'Medium (21-50mm)': 2, 'Large (>50mm)': 3,
    'Small': 1, 'Medium': 2, 'Large': 3
}

X_wbc = # YOUR CODE HERE

# Drop NaN
wbc_for_transfer = wbc_enriched.copy()
mask = X_wbc.notna().all(axis=1)
X_wbc = X_wbc[mask]
wbc_for_transfer = wbc_for_transfer[mask]

# Scale and predict
X_wbc_scaled = # YOUR CODE HERE -- use scaler.transform (NOT fit_transform!)
mortality_risk = # YOUR CODE HERE -- model.predict_proba(...)[:, 1]

wbc_for_transfer['mortality_risk_score'] = mortality_risk

print(f"Risk score range: {mortality_risk.min():.3f} to {mortality_risk.max():.3f}")
print(f"Mean risk score: {mortality_risk.mean():.3f}")

### 5.4 Analyze Results

In [ ]:
# TODO: Compare mortality risk scores between Malignant (0) and Benign (1) diagnosis
# Use groupby and describe()

# YOUR CODE HERE

In [ ]:
# TODO: Create a box plot of mortality_risk_score by diagnosis
# Hint: wbc_for_transfer.boxplot(column='mortality_risk_score', by='diagnosis')

# YOUR CODE HERE


plt.show()

In [ ]:
# TODO: Create a scatter plot of radius_mean vs mortality_risk_score
# Color by diagnosis (Malignant=0, Benign=1)
# This shows the relationship between cell morphology and predicted mortality

fig, ax = plt.subplots(figsize=(10, 6))

# YOUR CODE HERE


ax.set_xlabel('Mean Radius (FNA)')
ax.set_ylabel('SEER-Derived Mortality Risk Score')
ax.set_title('Cell Morphology vs. Population-Level Mortality Risk')
plt.tight_layout()
plt.show()

**Question:** Do malignant tumors (by cell morphology) tend to have higher SEER-derived mortality risk scores? What assumptions does this cross-dataset scoring rely on? Why is using a neutral grade value (2.0) for all Wisconsin patients a significant limitation?

*Your answer:*

---
# PART 6: Challenge Exercises
---

### 🚀 Challenge 1: Morphology Profile Comparison

Using Wisconsin BC data, compute a "morphology severity score" by averaging the z-scores of `radius_mean`, `concavity_mean`, `compactness_mean`, and `concave points_mean`. Then compare this score across your tumor size categories. Do worse morphology scores appear in categories with higher SEER mortality rates?

In [ ]:
# CHALLENGE 1: Morphology severity score

# YOUR CODE HERE

### 🚀 Challenge 2: Multi-Dimensional Lookup

Build a SEER lookup table grouped by **tumor_size_cat AND er_positive** (2 dimensions). Does enriching Wisconsin data by both tumor size and receptor status provide more granular clinical context?

In [ ]:
# CHALLENGE 2: Two-dimensional lookup

# YOUR CODE HERE

### 🚀 Challenge 3: Feature Importance from SEER Model

Using the trained LogisticRegression model, create a bar chart of feature coefficients. Which clinical features most strongly predict mortality? What does this suggest about the relative importance of tumor size, staging, grade, and receptor status?

In [ ]:
# CHALLENGE 3: Feature importance bar chart
# Hint: model.coef_[0] gives the coefficients
# train_features gives the feature names

# YOUR CODE HERE

### 🚀 Challenge 4: End-to-End Enrichment Pipeline

Create a single Wisconsin DataFrame that has:
1. Original Wisconsin BC columns (30 morphology features)
2. Population-level clinical context from Strategy 2 (mortality rate, receptor status, survival months)
3. Individual-level mortality risk score from Strategy 3

Then compute the correlation between `mortality_risk_score` and Wisconsin's top morphological features (`radius_mean`, `texture_mean`, `concavity_mean`, `compactness_mean`). Which morphological features best predict population-level mortality risk?

In [ ]:
# CHALLENGE 4: Full enrichment + correlation analysis

# YOUR CODE HERE

---
## ✅ Submission Checklist

Before submitting, make sure you have:

- [ ] Completed all TODO sections in Parts 1–5
- [ ] Run all cells to verify they work (Kernel → Restart & Run All)
- [ ] Answered all written questions
- [ ] Attempted at least 2 of the 4 challenge exercises
- [ ] Saved your notebook

---

## Quick Reference

| Task | Code |
|------|------|
| Load sklearn dataset | `load_breast_cancer()` then `pd.DataFrame(data.data, columns=data.feature_names)` |
| Load CSV | `pd.read_csv('file.csv')` |
| Strip column spaces | `df.columns = df.columns.str.strip()` |
| Map values | `df['col'].map({'A': 1, 'B': 0})` |
| Binning (equal width) | `pd.cut(df['col'], bins=[...], labels=[...])` |
| Binning (percentiles) | `pd.qcut(df['col'], q=3, labels=[...])` |
| Concat rows | `pd.concat([df1, df2], ignore_index=True)` |
| GroupBy + Agg | `df.groupby('col').agg(name=('col', 'func'))` |
| Merge | `df1.merge(df2, on='key', how='left')` |
| Scale features | `StandardScaler().fit_transform(X)` |
| Train model | `LogisticRegression().fit(X, y)` |
| Predict proba | `model.predict_proba(X)[:, 1]` |

---

## Remember Our Mantra:

# "One Patient, Many Datasets"

---

**See you in class!** 🎓